# 02D — Sentence Encoder Training, Full-Corpus Prediction, and Corpus Rebuild

**Pipeline stage:**

* load labeled sentence sample from 02C
* train sentence-level content classifier
* validate and choose operating threshold
* apply the model to the full sentence corpus
* retain only content sentences
* rebuild clean block-level and document-level corpora

**Output directory**

`output/02D_sentence_train/`

* `model/`
* `sentence_predictions/part_*.parquet`
* `content_sentences.parquet`
* `clean_ai_blocks.parquet`
* `clean_ai_docs.parquet`


In [1]:
!pip -q install pyarrow datasets transformers accelerate scikit-learn tqdm

## Environment Setup

In [2]:
import os
import gc
import math
import inspect
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from tqdm.auto import tqdm

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import datasets
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"

IN_DIR_02C = f"{BASE_DIR}/output/02C_sentence_sample"
OUT_DIR_02D = f"{BASE_DIR}/output/02D_sentence_train"

LABELED_SENTENCES_PARQUET = f"{IN_DIR_02C}/sentence_labels.parquet"
SENTENCES_DIR = f"{IN_DIR_02C}/sentences"

MODEL_DIR = f"{OUT_DIR_02D}/model"
PRED_DIR = f"{OUT_DIR_02D}/sentence_predictions"

CONTENT_SENTENCES_PARQUET = f"{OUT_DIR_02D}/content_sentences.parquet"
CLEAN_AI_BLOCKS_PARQUET = f"{OUT_DIR_02D}/clean_ai_blocks.parquet"
CLEAN_AI_DOCS_PARQUET = f"{OUT_DIR_02D}/clean_ai_docs.parquet"

os.makedirs(OUT_DIR_02D, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

assert os.path.exists(LABELED_SENTENCES_PARQUET), f"Missing: {LABELED_SENTENCES_PARQUET}"
assert os.path.exists(SENTENCES_DIR), f"Missing: {SENTENCES_DIR}"

## Configuration

In [4]:
SEED = 42
set_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NA"

USE_BF16 = bool(torch.cuda.is_available())
USE_FP16 = False

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)
print("gpu:", GPU_NAME)
print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("USE_BF16:", USE_BF16)

device: cuda
gpu: NVIDIA A100-SXM4-80GB
torch: 2.10.0+cu128
transformers: 5.0.0
USE_BF16: True


In [5]:
BACKBONE = "distilroberta-base"

MAX_LENGTH = 128
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

PER_DEVICE_TRAIN_BATCH = 64
PER_DEVICE_EVAL_BATCH = 128
GRAD_ACCUM = 1

INFER_BATCH_SIZE = 512
INFER_TABLE_BATCH_ROWS = 100_000

VALID_SIZE = 0.15

THRESHOLD_GRID = np.arange(0.20, 0.96, 0.05)

LENGTH_BUCKETS = [
    (0, 40),
    (41, 80),
    (81, 128),
    (129, 256),
    (257, 10_000_000),
]

## Load Labeled Sentence Sample

In [6]:
lab = pd.read_parquet(LABELED_SENTENCES_PARQUET)
print("raw labeled sentence shape:", lab.shape)

required_cols = [
    "sent_uid",
    "block_uid",
    "doc_id",
    "block_id",
    "sent_id",
    "sentence_text",
    "is_content_sentence",
    "label_status",
]
missing = [c for c in required_cols if c not in lab.columns]
assert not missing, f"Missing required columns: {missing}"

lab = lab[lab["label_status"] == "ok"].copy()
lab["text"] = lab["sentence_text"].fillna("").astype(str)
lab["label"] = pd.to_numeric(lab["is_content_sentence"], errors="coerce").fillna(0).astype(int)

print("usable labeled sentence rows:", len(lab))
print("\nlabel distribution:")
print(lab["label"].value_counts(dropna=False))

raw labeled sentence shape: (7302, 21)
usable labeled sentence rows: 7302

label distribution:
label
1    4729
0    2573
Name: count, dtype: int64


## Train / Validation Split

In [7]:
train_df, valid_df = train_test_split(
    lab[["sent_uid", "text", "label"]].copy(),
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=lab["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)

print("\ntrain label distribution:")
print(train_df["label"].value_counts(dropna=False))

print("\nvalid label distribution:")
print(valid_df["label"].value_counts(dropna=False))

train shape: (6206, 3)
valid shape: (1096, 3)

train label distribution:
label
1    4019
0    2187
Name: count, dtype: int64

valid label distribution:
label
1    710
0    386
Name: count, dtype: int64


## Utility Functions

In [8]:
def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[["text", "label"]], preserve_index=False)

def tokenize_batch(examples: Dict[str, List[str]], tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

def softmax_pos_prob(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    ex = np.exp(x)
    p = ex / ex.sum(axis=1, keepdims=True)
    return p[:, 1]

def compute_binary_metrics(y_true: np.ndarray, p_pos: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (p_pos >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "pred_positive_rate": float(y_pred.mean()),
        "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
        "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
        "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
    }

def search_best_threshold(y_true: np.ndarray, p_pos: np.ndarray, grid: np.ndarray) -> pd.DataFrame:
    rows = []
    for t in grid:
        rows.append(compute_binary_metrics(y_true, p_pos, float(t)))
    return pd.DataFrame(rows).sort_values(["f1", "accuracy"], ascending=[False, False]).reset_index(drop=True)

def compute_metrics_default(eval_pred):
    logits, labels = eval_pred
    probs = softmax_pos_prob(np.asarray(logits))
    y_pred = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, y_pred)
    return {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
    }

def build_training_args(output_dir: str) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters.keys())

    kwargs = {
        "output_dir": output_dir,
        "num_train_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "save_strategy": "no",
        "report_to": "none",
        "fp16": USE_FP16,
        "bf16": USE_BF16,
        "remove_unused_columns": True,
    }

    if "eval_strategy" in valid:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in valid:
        kwargs["evaluation_strategy"] = "epoch"

    kwargs = {k: v for k, v in kwargs.items() if k in valid}
    return TrainingArguments(**kwargs)

def sanity_forward_pass(model, tokenizer, texts: List[str], labels: List[int]) -> None:
    model = model.to(DEVICE).eval()
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    enc["labels"] = torch.tensor(labels, dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        out = model(**enc)
        print("logits finite:", torch.isfinite(out.logits).all().item())
        print("loss finite:", torch.isfinite(out.loss).item(), "| loss:", float(out.loss.detach().cpu().item()))

def infer_probs(model, tokenizer, texts: List[str], batch_size: int) -> np.ndarray:
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = (
                torch.softmax(logits, dim=1)[:, 1]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )
            out.append(probs)
    return np.concatenate(out) if out else np.array([], dtype=np.float32)

def bucket_indices_by_length(lengths: np.ndarray, buckets: List[Tuple[int, int]]) -> List[np.ndarray]:
    idx_groups = []
    for lo, hi in buckets:
        idx = np.where((lengths >= lo) & (lengths <= hi))[0]
        if len(idx) > 0:
            idx_groups.append(idx)
    return idx_groups

## Tokenizer and Model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=2)

sanity_forward_pass(
    model,
    tokenizer,
    texts=train_df["text"].head(4).tolist(),
    labels=train_df["label"].head(4).tolist(),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


logits finite: True
loss finite: True | loss: 0.7732946276664734


## Dataset Preparation

In [10]:
ds_train = make_hf_dataset(train_df).map(lambda x: tokenize_batch(x, tokenizer), batched=True)
ds_valid = make_hf_dataset(valid_df).map(lambda x: tokenize_batch(x, tokenizer), batched=True)

ds_train = ds_train.remove_columns(["text"]).rename_column("label", "labels")
ds_valid = ds_valid.remove_columns(["text"]).rename_column("label", "labels")

ds_train = ds_train.cast_column("labels", datasets.Value("int64"))
ds_valid = ds_valid.cast_column("labels", datasets.Value("int64"))

collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/6206 [00:00<?, ? examples/s]

Map:   0%|          | 0/1096 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/6206 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1096 [00:00<?, ? examples/s]

## Training

In [11]:
training_args = build_training_args(output_dir=OUT_DIR_02D)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_valid,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_default,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.426313,0.161876,0.937044,0.936054,0.969014,0.952249
2,0.161180,0.153561,0.944343,0.948824,0.966197,0.957432
3,0.128195,0.162190,0.942518,0.933066,0.981690,0.956760


TrainOutput(global_step=291, training_loss=0.20089034935862748, metrics={'train_runtime': 12.603, 'train_samples_per_second': 1477.264, 'train_steps_per_second': 23.09, 'total_flos': 577692912792696.0, 'train_loss': 0.20089034935862748, 'epoch': 3.0})

## Validation and Threshold Selection

In [12]:
pred_valid = trainer.predict(ds_valid)
p_valid = softmax_pos_prob(np.asarray(pred_valid.predictions))
y_valid = np.asarray(pred_valid.label_ids)

threshold_audit = search_best_threshold(y_valid, p_valid, THRESHOLD_GRID)
display(threshold_audit)

best_threshold = float(threshold_audit.iloc[0]["threshold"])
print("selected threshold:", best_threshold)

valid_pred_label = (p_valid >= best_threshold).astype(int)

valid_predictions = valid_df.copy()
valid_predictions["p_content_sentence"] = p_valid
valid_predictions["pred_content_sentence"] = valid_pred_label

print("\nvalidation prediction distribution:")
print(valid_predictions["pred_content_sentence"].value_counts(dropna=False))

display(valid_predictions.head(20))

,threshold,accuracy,precision,recall,f1,pred_positive_rate,tn,fp,fn,tp
0,0.70,0.947080,0.949036,0.970423,0.959610,0.662409,349,37,21,689
1,0.65,0.946168,0.946502,0.971831,0.958999,0.665146,347,39,20,690
2,0.80,0.946168,0.956522,0.960563,0.958538,0.650547,355,31,28,682
3,0.85,0.946168,0.963016,0.953521,0.958245,0.641423,360,26,33,677
4,0.75,0.944343,0.950069,0.964789,0.957372,0.657847,350,36,25,685
5,0.60,0.943431,0.942623,0.971831,0.957004,0.667883,344,42,20,690
6,0.50,0.942518,0.933066,0.981690,0.956760,0.681569,336,50,13,697
7,0.55,0.942518,0.937754,0.976056,0.956522,0.674270,340,46,17,693
8,0.45,0.941606,0.930667,0.983099,0.956164,0.684307,334,52,12,698
9,0.35,0.939781,0.925926,0.985915,0.954980,0.689781,330,56,10,700


selected threshold: 0.7

validation prediction distribution:
pred_content_sentence
1    726
0    370
Name: count, dtype: int64


,sent_uid,text,label,p_content_sentence,pred_content_sentence
0,141141:10:56,"Just as the term goes ""don't use AI for the sa...",1,0.996676,1
1,35765:12:0,We had the opportunity to be one of the first ...,1,0.993307,1
2,83552:0:4,Jerz's Literacy Weblog (est.,0,0.005820,0
3,143353:23:5,It was “just better to part ways on good terms...,1,0.989511,1
4,155463:24:12,"Developers are juggling multiple tools, langua...",1,0.996727,1
5,194307:1:15,Leading AI Network trading platforms typically...,1,0.996350,1
6,133905:4:11,Many travel companies have released AI-powered...,1,0.997917,1
7,19833:0:1,Yahoo Finance Yahoo Finance Sign in Mail Sign ...,0,0.004468,0
8,180004:3:16,The AI Foundation Model continuously learns fr...,1,0.994940,1
9,144147:11:9,"""When you share data freely today, you're cont...",1,0.995245,1


## Save Final Model

In [13]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("wrote:", MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/model


## Full-Corpus Sentence Prediction

In [14]:
tokenizer_inf = AutoTokenizer.from_pretrained(MODEL_DIR)
model_inf = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()

sentence_files = sorted(
    [
        os.path.join(SENTENCES_DIR, f)
        for f in os.listdir(SENTENCES_DIR)
        if f.endswith(".parquet")
    ]
)

assert sentence_files, f"No sentence shards found in: {SENTENCES_DIR}"
print("sentence shard count:", len(sentence_files))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

sentence shard count: 27


In [15]:
for shard_path in tqdm(sentence_files, desc="Sentence shards", position=0):
    shard_name = os.path.basename(shard_path).replace(".parquet", "")
    pf = pq.ParquetFile(shard_path)
    n_parts = math.ceil(pf.metadata.num_rows / INFER_TABLE_BATCH_ROWS)

    for part_j, batch in tqdm(
        enumerate(
            pf.iter_batches(
                batch_size=INFER_TABLE_BATCH_ROWS,
                columns=[
                    "doc_id",
                    "block_id",
                    "block_uid",
                    "sent_id",
                    "sent_uid",
                    "url",
                    "date",
                    "language",
                    "title",
                    "domain",
                    "sentence_text",
                    "sent_len",
                    "sent_len_bucket",
                    "suspect_flag",
                    "p_ai_block",
                    "pred_ai_block",
                ],
            )
        ),
        total=n_parts,
        desc=shard_name,
        position=1,
        leave=False,
    ):
        out_path = os.path.join(PRED_DIR, f"{shard_name}_part{part_j:03d}_pred.parquet")

        if os.path.exists(out_path):
            continue

        pdf = batch.to_pandas()
        pdf["sentence_text"] = pdf["sentence_text"].fillna("").astype(str)

        if "sent_len" not in pdf.columns:
            pdf["sent_len"] = pdf["sentence_text"].str.len().astype(int)

        texts = pdf["sentence_text"].tolist()
        lengths = pdf["sent_len"].to_numpy()

        p_content = np.zeros(len(pdf), dtype=np.float32)
        pred_content = np.zeros(len(pdf), dtype=np.int8)

        idx_groups = bucket_indices_by_length(lengths, LENGTH_BUCKETS)

        for idx in idx_groups:
            sub_texts = [texts[k] for k in idx]
            p_sub = infer_probs(
                model_inf,
                tokenizer_inf,
                sub_texts,
                batch_size=INFER_BATCH_SIZE,
            )
            p_content[idx] = p_sub
            pred_content[idx] = (p_sub >= best_threshold).astype(np.int8)

        pdf["p_content_sentence"] = p_content
        pdf["pred_content_sentence"] = pred_content

        pdf.to_parquet(out_path, index=False)

        del pdf
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("prediction shards written:", PRED_DIR)

Sentence shards:   0%|          | 0/27 [00:00<?, ?it/s]

part_0000:   0%|          | 0/3 [00:00<?, ?it/s]

part_0001:   0%|          | 0/3 [00:00<?, ?it/s]

part_0002:   0%|          | 0/3 [00:00<?, ?it/s]

part_0003:   0%|          | 0/3 [00:00<?, ?it/s]

...

part_0023:   0%|          | 0/3 [00:00<?, ?it/s]

part_0024:   0%|          | 0/3 [00:00<?, ?it/s]

part_0025:   0%|          | 0/3 [00:00<?, ?it/s]

part_0026:   0%|          | 0/1 [00:00<?, ?it/s]

prediction shards written: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/sentence_predictions


## Build Retained Content Sentence Table

In [16]:
pred_files = sorted(
    [
        os.path.join(PRED_DIR, f)
        for f in os.listdir(PRED_DIR)
        if f.endswith(".parquet")
    ]
)

assert pred_files, f"No prediction shards found in: {PRED_DIR}"

writer = None
rows_written = 0

for fpath in tqdm(pred_files, desc="Collecting content sentences"):
    part = pd.read_parquet(fpath)
    keep = part[part["pred_content_sentence"] == 1].copy()

    if len(keep) == 0:
        continue

    table = pa.Table.from_pandas(keep, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(
            CONTENT_SENTENCES_PARQUET,
            table.schema,
            compression="zstd",
        )

    writer.write_table(table)
    rows_written += len(keep)

    del part, keep, table
    gc.collect()

if writer is not None:
    writer.close()

print("wrote:", CONTENT_SENTENCES_PARQUET)
print("content sentence rows:", rows_written)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/content_sentences.parquet
content sentence rows: 4586024


## Rebuild Clean AI Blocks

In [17]:
content_sentences = pd.read_parquet(CONTENT_SENTENCES_PARQUET)
print("content_sentences shape:", content_sentences.shape)

required_rebuild_cols = [
    "doc_id",
    "block_id",
    "block_uid",
    "sent_id",
    "url",
    "date",
    "language",
    "title",
    "domain",
    "sentence_text",
    "p_ai_block",
]
missing = [c for c in required_rebuild_cols if c not in content_sentences.columns]
assert not missing, f"Missing required columns for rebuild: {missing}"

content_sentences = content_sentences.sort_values(
    ["doc_id", "block_id", "sent_id"],
    kind="mergesort"
).reset_index(drop=True)

clean_blocks = (
    content_sentences.groupby(
        ["doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain"],
        dropna=False,
        as_index=False
    )
    .agg(
        clean_block_text=("sentence_text", lambda x: " ".join([str(v).strip() for v in x if str(v).strip()])),
        n_content_sentences=("sentence_text", "size"),
        p_ai_block=("p_ai_block", "max"),
    )
)

clean_blocks["clean_block_text"] = clean_blocks["clean_block_text"].fillna("").astype(str).str.strip()
clean_blocks["clean_block_len"] = clean_blocks["clean_block_text"].str.len().astype(int)

clean_blocks = clean_blocks[clean_blocks["clean_block_len"] > 0].copy().reset_index(drop=True)

clean_blocks.to_parquet(CLEAN_AI_BLOCKS_PARQUET, index=False)
print("wrote:", CLEAN_AI_BLOCKS_PARQUET)
print("clean_blocks shape:", clean_blocks.shape)

content_sentences shape: (4586024, 18)
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/clean_ai_blocks.parquet
clean_blocks shape: (731989, 12)


## Rebuild Clean AI Documents

In [18]:
clean_blocks = clean_blocks.sort_values(["doc_id", "block_id"], kind="mergesort").reset_index(drop=True)

clean_docs = (
    clean_blocks.groupby(
        ["doc_id", "url", "date", "language", "title", "domain"],
        dropna=False,
        as_index=False
    )
    .agg(
        clean_ai_doc_text=("clean_block_text", lambda x: "\n\n".join([str(v).strip() for v in x if str(v).strip()])),
        n_clean_blocks=("block_id", "nunique"),
        n_content_sentences=("n_content_sentences", "sum"),
        max_p_ai_block=("p_ai_block", "max"),
    )
)

clean_docs["clean_ai_doc_text"] = clean_docs["clean_ai_doc_text"].fillna("").astype(str).str.strip()
clean_docs["clean_ai_doc_len"] = clean_docs["clean_ai_doc_text"].str.len().astype(int)

clean_docs = clean_docs[clean_docs["clean_ai_doc_len"] > 0].copy().reset_index(drop=True)

clean_docs.to_parquet(CLEAN_AI_DOCS_PARQUET, index=False)
print("wrote:", CLEAN_AI_DOCS_PARQUET)
print("clean_docs shape:", clean_docs.shape)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/clean_ai_docs.parquet
clean_docs shape: (181008, 11)


## Quick Readout

In [19]:
content_sentences_preview = pd.read_parquet(CONTENT_SENTENCES_PARQUET)
clean_blocks_preview = pd.read_parquet(CLEAN_AI_BLOCKS_PARQUET)
clean_docs_preview = pd.read_parquet(CLEAN_AI_DOCS_PARQUET)

print("content_sentences shape:", content_sentences_preview.shape)
print("clean_blocks shape:", clean_blocks_preview.shape)
print("clean_docs shape:", clean_docs_preview.shape)

print("\ncontent sentence prediction distribution:")
print(content_sentences_preview["pred_content_sentence"].value_counts(dropna=False))

print("\nclean block length summary:")
display(clean_blocks_preview["clean_block_len"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]))

print("\nclean document length summary:")
display(clean_docs_preview["clean_ai_doc_len"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]))

print("\nexample clean blocks:")
display(
    clean_blocks_preview[
        [
            "doc_id",
            "block_id",
            "domain",
            "title",
            "n_content_sentences",
            "clean_block_text",
        ]
    ].head(15)
)

print("\nexample clean documents:")
display(
    clean_docs_preview[
        [
            "doc_id",
            "domain",
            "title",
            "n_clean_blocks",
            "n_content_sentences",
            "clean_ai_doc_text",
        ]
    ].head(10)
)

content_sentences shape: (4586024, 18)
clean_blocks shape: (731989, 12)
clean_docs shape: (181008, 11)

content sentence prediction distribution:
pred_content_sentence
1    4586024
Name: count, dtype: int64

clean block length summary:


,clean_block_len
count,731989.000000
mean,943.715279
std,1571.435531
min,15.000000
1%,36.000000
10%,65.000000
50%,291.000000
90%,2908.000000
99%,7510.000000
max,19813.000000



clean document length summary:


,clean_ai_doc_len
count,181008.000000
mean,3822.434174
std,2702.512784
min,15.000000
1%,84.000000
10%,999.000000
50%,3331.000000
90%,7039.000000
99%,12387.000000
max,48067.000000



example clean blocks:


,doc_id,block_id,domain,title,n_content_sentences,clean_block_text
0,0,0,blockworks.co,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",14,The price of BAD is down -0.61% since last hou...
1,1,7,boingboing.net,This AI video of gymnastics might be the freak...,6,I'm sure by now you're tired of looking at ter...
2,1,8,boingboing.net,This AI video of gymnastics might be the freak...,5,Imagine showing these to a tribe that's never ...
3,1,11,boingboing.net,This AI video of gymnastics might be the freak...,1,This AI video attempt to show gymnastics is on...
4,2,6,boingboing.net,"If using AI feels like a chore, try this - Boi...",6,AI isn't going anywhere. The problem is that w...
5,2,7,boingboing.net,"If using AI feels like a chore, try this - Boi...",11,"Instead, 1minAI bundles everything you need in..."
6,2,8,boingboing.net,"If using AI feels like a chore, try this - Boi...",1,"To play this human claw machine, you get strap..."
7,3,4,citylife.capetown,The Road Ahead: How China's AI Foundation Mode...,22,o modelo da Fundación AI de China está a dar f...
8,4,0,citylife.capetown,Microsoft and Nvidia to Empower Developers wit...,1,Microsoft and Nvidia to Empower Developers wit...
9,4,4,citylife.capetown,Microsoft and Nvidia to Empower Developers wit...,25,"Қараша 15, 2023 Microsoft and Nvidia are joini..."



example clean documents:


,doc_id,domain,title,n_clean_blocks,n_content_sentences,clean_ai_doc_text
0,0,blockworks.co,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",1,14,The price of BAD is down -0.61% since last hou...
1,1,boingboing.net,This AI video of gymnastics might be the freak...,3,12,I'm sure by now you're tired of looking at ter...
2,2,boingboing.net,"If using AI feels like a chore, try this - Boi...",3,18,AI isn't going anywhere. The problem is that w...
3,3,citylife.capetown,The Road Ahead: How China's AI Foundation Mode...,1,22,o modelo da Fundación AI de China está a dar f...
4,4,citylife.capetown,Microsoft and Nvidia to Empower Developers wit...,2,26,Microsoft and Nvidia to Empower Developers wit...
5,5,citylife.capetown,Google Releases New Chatbot Bard as a Strong C...,2,15,Google Releases New Chatbot Bard as a Strong C...
6,6,citylife.capetown,Zoom Expands AI Offering with AI Companion and...,3,16,Zoom Expands AI Offering with AI Companion and...
7,7,citylife.capetown,Pro-AI Thinking: Enhancing Industrial Environm...,8,27,Pro-AI Thinking: Enhancing Industrial Environm...
8,8,clickup.com,Best AI Prompts for Business Risk Management,1,35,Harness the power of ClickUp AI to make smarte...
9,9,crooksandliars.com,State AGs Warn AI Companies: Clean Up Your Chi...,1,16,"By Susie Madrak — December 15, 2025 The parent..."
